In [1]:
import pandas as pd

In [2]:
df_ground_truth = pd.read_csv('ground_truth-new_alex.csv')
len(df_ground_truth)

395

In [3]:
df_ground_truth.head()

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf


In [4]:
# convert the df to dict
ground_truth = df_ground_truth.to_dict(orient='records')
ground_truth[:3]

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'}]

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
# A boost in a search index is a multiplier applied to a document's relevance score
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](h

In [7]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [8]:
q = ground_truth[0]
q = q['question']
q

'Is it okay to join the course late if I just found it now?'

In [9]:
results = text_search(q)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate or discuss a larger idea with other students, but each submitted project must stand on its own. A shared system can work only if it is clearly decomposed into independent projects, where each person has a separate qualifying component and a separate repository.\n\nIf the work cannot be evaluated independently for each participant, it does not satisfy the project requirement.'},
 {'id': '610ccb23c0',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'What happens to code s

In [10]:
doc_id = ground_truth[0]['document']
doc_id

'74eb249bbf'

In [11]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
acf8fa5356 == 74eb249bbf: False


In [12]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance
# relevance matrix - In information retrieval, a relevance matrix (or scoring matrix) is the mathematical foundation search engines use 
# to quantify how well documents or web pages answer a user's search query.
# It translates unstructured text and queries into numerical representations to calculate the most accurate results.

[1, 0, 0, 0, 0]

In [13]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [14]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

Is it okay to join the course late if I just found it now?


[1, 0, 0, 0, 0]

In [15]:
q = ground_truth[10]
print(q["question"])
compute_relevance_text(q)

How do I join the Office Hours or live workshop if I don’t have the Zoom link?


[1, 0, 0, 0, 0]

In [16]:
from tqdm.auto import tqdm

In [17]:
# In search and AI (like RAG), Hit Rate (often Hit@K) measures 
# how often the system successfully finds at least one relevant result within the top K retrieved items. 
# It is a simple binary metric (1 if a relevant item is in the top results, 0 if not) used to evaluate retriever accuracy.

In [18]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [19]:
relevance = compute_relevance_total(ground_truth, text_search)
relevance

  0%|          | 0/395 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0,

In [20]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

hit_rate(relevance)

0.6506329113924051

In [21]:
# mean reciprocal rank
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

mrr(relevance)

0.5736286919831224

In [22]:
# mrr is better than hit rate as metric

In [23]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

evaluate(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': 0.5736286919831224}

In [24]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

evaluate(ground_truth, text_search_v2)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6632911392405063, 'mrr': 0.5800000000000001}

In [25]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [26]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [27]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
34,5.0,10.0,0.2,0.754430,0.652110
18,2.0,4.0,0.1,0.754430,0.651688
19,2.0,4.0,0.2,0.749367,0.649916
35,5.0,10.0,0.5,0.749367,0.649916
3,1.0,2.0,0.1,0.749367,0.649916
33,5.0,10.0,0.1,0.749367,0.649620
20,2.0,4.0,0.5,0.746835,0.648101
4,1.0,2.0,0.2,0.746835,0.647679
7,1.0,4.0,0.2,0.754430,0.636287
6,1.0,4.0,0.1,0.754430,0.635781
